# Variational thermal-state preparation

This notebook prepares approximate thermal states for the 4x4 transverse-field Ising model. The workflow is:

1. Represent a classical distribution over computational-basis states.
2. Transform each sampled basis state with a trainable quantum ansatz.
3. Optimize the Helmholtz free energy at each challenge temperature.
4. Sample the optimized states and write the submission files.

The comments below explain both what each step does and why it is needed.

## Depolarizing-noise configuration

The energy and submission circuits use PennyLane's `default.mixed` device so that depolarizing channels can be represented as density matrices. Set `USE_NOISE` to `False` to run the same circuit noiselessly.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as np

# The challenge uses a 4x4 lattice and g=1.
NUM_QUBITS = 16
N_LAYERS = 2
FIELD_STRENGTH = 1.0
TEMPERATURES = (1.1875, 2.375, 4.0)
N_SHOTS = 128

wires = list(range(NUM_QUBITS))

# Set USE_NOISE = False for a noiseless density-matrix simulation.
USE_NOISE = True
P_ONE_QUBIT = 1e-5 if USE_NOISE else 0.0
P_TWO_QUBIT = 1.4e-4 if USE_NOISE else 0.0

dev = qml.device("default.mixed", wires=wires)

## 1. Classical latent distribution

The latent model assigns an independent probability to each qubit being 1. Its logits define a probability for every one of the $2^{16}$ computational-basis states. This gives us a simple, differentiable entropy model while keeping state preparation easy: sample a bitstring, prepare it, and apply the quantum ansatz.


In [2]:
def latent_probability_distribution(logits):
    """Return the product distribution over all computational-basis states."""
    qubit_probabilities = 1.0 / (1.0 + np.exp(-logits))
    state_indices = np.arange(2 ** NUM_QUBITS)
    bitstrings = np.array([
        [(int(index) >> bit) & 1 for bit in range(NUM_QUBITS - 1, -1, -1)]
        for index in state_indices
    ], dtype=float)

    probabilities = np.prod(
        bitstrings * qubit_probabilities
        + (1.0 - bitstrings) * (1.0 - qubit_probabilities),
        axis=1,
    )
    return probabilities / np.sum(probabilities)


def index_to_bitstring(index):
    """Convert an integer basis-state index to the wire-order bitstring."""
    return np.array([
        (int(index) >> bit) & 1
        for bit in range(NUM_QUBITS - 1, -1, -1)
    ])

## 2. Quantum processing ansätze

The notebook keeps the original circuit choices and adds two more expressive variants:

- **Contiguous hardware-efficient ansatz:** rotations followed by a chain of CNOTs.
- **Interleaved hardware-efficient ansatz:** rotations followed by even and odd CNOT sublayers.
- **Normal HVA:** shared $ZZ$ and $RX$ angles based on the Ising Hamiltonian.
- **Independent HVA:** a separate parameter for every lattice interaction and single-qubit rotation.
- **Fourier independent HVA:** independent-HVA gate families whose angles are generated by 2D Fourier modes.

Change `ANSATZ_NAME` in the next code cell to compare the choices. The optimization and submission code below automatically uses the matching parameter count.


In [3]:
# Select the circuit used by the optimization and submission cells.
ANSATZ_NAME = "interleaved"
# (contiguous, interleaved, hva, independent_hva, fourier_independent_hva)

FOURIER_MODES = 1
COEFFICIENTS_PER_FAMILY = 1 + 4 * FOURIER_MODES


def _check_square_lattice(ansatz_wires):
    side = int(np.sqrt(len(ansatz_wires)))
    if side * side != len(ansatz_wires):
        raise ValueError("The number of wires must form a square lattice.")
    return side


def contiguous_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """Hardware-efficient ansatz with a contiguous CNOT chain."""
    for layer in range(n_layers):
        theta = params[layer]
        for wire in ansatz_wires:
            qml.RZ(theta, wires=wire)
            qml.RY(theta, wires=wire)
        for start in range(len(ansatz_wires) - 1):
            qml.CNOT(wires=[ansatz_wires[start], ansatz_wires[start + 1]])


def interleaved_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """Hardware-efficient ansatz with even and odd CNOT sublayers."""
    for layer in range(n_layers):
        theta = params[layer]
        for wire in ansatz_wires:
            qml.RZ(theta, wires=wire)
            qml.RY(theta, wires=wire)
        for start in range(0, len(ansatz_wires) - 1, 2):
            qml.CNOT(wires=[ansatz_wires[start], ansatz_wires[start + 1]])
        for start in range(1, len(ansatz_wires) - 1, 2):
            qml.CNOT(wires=[ansatz_wires[start], ansatz_wires[start + 1]])


def hva_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """Hamiltonian variational ansatz with shared ZZ and RX angles."""
    side = _check_square_lattice(ansatz_wires)
    for layer in range(n_layers):
        theta = params[layer]
        for row in range(side):
            for column in range(side):
                wire = row * side + column
                right = row * side + (column + 1) % side
                down = ((row + 1) % side) * side + column
                qml.IsingZZ(theta, wires=[ansatz_wires[wire], ansatz_wires[right]])
                qml.IsingZZ(theta, wires=[ansatz_wires[wire], ansatz_wires[down]])
        for wire in ansatz_wires:
            qml.RX(theta, wires=wire)


def independent_hva_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """HVA with an independent angle for every ZZ interaction and RX gate."""
    side = _check_square_lattice(ansatz_wires)
    interactions_per_layer = 2 * len(ansatz_wires)
    parameters_per_layer = interactions_per_layer + len(ansatz_wires)

    for layer in range(n_layers):
        start = layer * parameters_per_layer
        zz_params = params[start:start + interactions_per_layer]
        rx_params = params[start + interactions_per_layer:start + parameters_per_layer]
        zz_index = 0
        for row in range(side):
            for column in range(side):
                wire = row * side + column
                right = row * side + (column + 1) % side
                down = ((row + 1) % side) * side + column
                qml.IsingZZ(zz_params[zz_index], wires=[ansatz_wires[wire], ansatz_wires[right]])
                qml.IsingZZ(zz_params[zz_index + 1], wires=[ansatz_wires[wire], ansatz_wires[down]])
                zz_index += 2
        for wire, angle in zip(ansatz_wires, rx_params):
            qml.RX(angle, wires=wire)


def fourier_lattice_angle(coefficients, wire, side):
    """Evaluate a 2D Fourier expansion at one lattice site."""
    row, column = divmod(wire, side)
    angle = coefficients[0]
    offset = 1
    for mode in range(1, FOURIER_MODES + 1):
        angle += coefficients[offset] * np.cos(2 * np.pi * mode * column / side)
        angle += coefficients[offset + 1] * np.sin(2 * np.pi * mode * column / side)
        angle += coefficients[offset + 2] * np.cos(2 * np.pi * mode * row / side)
        angle += coefficients[offset + 3] * np.sin(2 * np.pi * mode * row / side)
        offset += 4
    return angle


def fourier_independent_hva_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """Independent HVA whose ZZ and RX angles use lattice Fourier modes."""
    side = _check_square_lattice(ansatz_wires)
    parameters_per_layer = 3 * COEFFICIENTS_PER_FAMILY

    for layer in range(n_layers):
        start = layer * parameters_per_layer
        zz_horizontal = params[start:start + COEFFICIENTS_PER_FAMILY]
        zz_vertical = params[start + COEFFICIENTS_PER_FAMILY:start + 2 * COEFFICIENTS_PER_FAMILY]
        rx = params[start + 2 * COEFFICIENTS_PER_FAMILY:start + parameters_per_layer]

        for row in range(side):
            for column in range(side):
                wire = row * side + column
                right = row * side + (column + 1) % side
                down = ((row + 1) % side) * side + column
                qml.IsingZZ(
                    fourier_lattice_angle(zz_horizontal, wire, side),
                    wires=[ansatz_wires[wire], ansatz_wires[right]],
                )
                qml.IsingZZ(
                    fourier_lattice_angle(zz_vertical, wire, side),
                    wires=[ansatz_wires[wire], ansatz_wires[down]],
                )
        for wire in ansatz_wires:
            qml.RX(fourier_lattice_angle(rx, wire, side), wires=wire)


ANSATZ_FUNCTIONS = {
    "contiguous": contiguous_ansatz,
    "interleaved": interleaved_ansatz,
    "hva": hva_ansatz,
    "independent_hva": independent_hva_ansatz,
    "fourier_independent_hva": fourier_independent_hva_ansatz,
}


def ansatz_parameter_count(name=ANSATZ_NAME, n_layers=N_LAYERS):
    """Return the number of circuit parameters for a selected ansatz."""
    if name in {"contiguous", "interleaved", "hva"}:
        return n_layers
    if name == "independent_hva":
        return n_layers * 3 * NUM_QUBITS
    if name == "fourier_independent_hva":
        return n_layers * 3 * COEFFICIENTS_PER_FAMILY
    raise ValueError(f"Unknown ansatz: {name}")


def variational_ansatz(params, ansatz_wires, n_layers=N_LAYERS):
    """Dispatch to the ansatz selected by ``ANSATZ_NAME``."""
    ANSATZ_FUNCTIONS[ANSATZ_NAME](params, ansatz_wires, n_layers)


# Insert gate-dependent depolarizing channels after circuit operations.
one_qubit_operations = qml.noise.op_in([
    qml.RZ,
    qml.RY,
    qml.RX,
    qml.Hadamard,
])
two_qubit_operations = qml.noise.op_in([qml.IsingZZ, qml.CNOT])

noise_model = qml.NoiseModel({
    one_qubit_operations: qml.noise.partial_wires(
        qml.DepolarizingChannel, P_ONE_QUBIT
    ),
    two_qubit_operations: qml.noise.partial_wires(
        qml.DepolarizingChannel, P_TWO_QUBIT
    ),
})


## 3. Hamiltonian and variational objective

The target Hamiltonian is

$$H = -J \sum_{\langle i,j \rangle} Z_i Z_j - g \sum_i X_i,$$

with $J=g=1$ and periodic boundary conditions. We minimize

$$F = \langle H \rangle - \frac{S}{\beta}, \qquad S=-\sum_x p(x)\log p(x),$$

where $\beta=1/T$. Lower free energy means a better approximation to the thermal state.

In [ ]:
def transverse_ising_hamiltonian(g=FIELD_STRENGTH):
    """Build the 4x4 transverse-field Ising Hamiltonian."""
    side = int(np.sqrt(NUM_QUBITS))
    coefficients = []
    observables = []

    for row in range(side):
        for column in range(side):
            wire = row * side + column
            right = row * side + (column + 1) % side
            down = ((row + 1) % side) * side + column
            coefficients.extend((-1.0, -1.0))
            observables.extend((qml.PauliZ(wire) @ qml.PauliZ(right),
                                qml.PauliZ(wire) @ qml.PauliZ(down)))

    for wire in wires:
        coefficients.append(-g)
        observables.append(qml.PauliX(wire))

    return qml.Hamiltonian(coefficients, observables)


def helmholtz_free_energy(params, beta, n_energy_samples=32, seed=1234):
    """Estimate the free energy for one set of latent and circuit parameters."""
    latent_logits = params[:NUM_QUBITS]
    circuit_params = params[NUM_QUBITS:]
    hamiltonian = transverse_ising_hamiltonian()

    @qml.noise.add_noise(noise_model=noise_model)
    @qml.qnode(dev, interface="autograd")
    def energy_circuit(theta, bitstring):
        qml.BasisState(bitstring, wires=wires)
        variational_ansatz(theta, wires)
        return qml.expval(hamiltonian)

    probabilities = latent_probability_distribution(latent_logits)
    rng = np.random.default_rng(seed)
    sampled_indices = rng.choice(
        2 ** NUM_QUBITS, size=n_energy_samples, replace=True,
        p=np.asarray(qml.math.detach(probabilities)),
    )

    energies = [energy_circuit(circuit_params, index_to_bitstring(index))
                for index in sampled_indices]
    expected_energy = sum(energies) / len(energies)

    safe_probabilities = np.clip(probabilities, 1e-12, 1.0)
    entropy = -np.sum(safe_probabilities * np.log(safe_probabilities))
    return expected_energy - entropy / beta

: 

## 4. Optimize one state per temperature

The optimizer starts from the same neutral initialization at each temperature. We keep the full history because the final optimizer step is not necessarily the best point when the objective is estimated by sampling. The selected parameters are therefore the history entry with the lowest re-evaluated free energy.

In [ ]:
N_CIRCUIT_PARAMS = ansatz_parameter_count()
INITIAL_PARAMS = np.concatenate([
    np.zeros(NUM_QUBITS, requires_grad=True),
    np.full(N_CIRCUIT_PARAMS, 0.1, requires_grad=True),
])
MAX_ITERATIONS = 10
CONVERGENCE_TOLERANCE = 1e-6
LEARNING_RATE = 0.01

optimized_params = {}
free_energy_history = {}

for temperature in TEMPERATURES:
    beta = 1.0 / temperature
    optimizer = qml.AdamOptimizer(stepsize=LEARNING_RATE)
    params = np.array(INITIAL_PARAMS, copy=True)
    history = [params]

    print(f"\nOptimizing T={temperature}")
    for iteration in range(MAX_ITERATIONS):
        params, previous_energy = optimizer.step_and_cost(
            lambda current: helmholtz_free_energy(current, beta),
            params,
        )
        history.append(params)

        if iteration % 5 == 0:
            current_energy = helmholtz_free_energy(params, beta)
            change = abs(current_energy - previous_energy)
            print(f"  iteration={iteration}, free energy={current_energy:.8f}, change={change:.2e}")
            if change <= CONVERGENCE_TOLERANCE:
                break

    energies = np.array([
        float(helmholtz_free_energy(candidate, beta))
        for candidate in history
    ])
    best_index = int(np.argmin(energies))
    optimized_params[temperature] = np.asarray(history[best_index])
    free_energy_history[temperature] = energies
    print(f"  selected iteration={best_index}, free energy={energies[best_index]:.10f}")


Optimizing T=1.1875


## 5. Generate the challenge submissions

For every temperature and basis, we draw exactly 128 latent bitstrings and measure the resulting quantum states. The basis rotations convert X and Y measurements into the device's computational (Z) basis. Each JSON file is ready for the challenge scorer.

In [ ]:
submission_dev = qml.device("default.mixed", wires=wires)

@qml.noise.add_noise(noise_model=noise_model)
@qml.qnode(submission_dev, shots=1)
def sample_state(circuit_params, bitstring, basis):
    qml.BasisState(bitstring, wires=wires)
    variational_ansatz(circuit_params, wires)

    if basis == "X":
        for wire in wires:
            qml.Hadamard(wires=wire)
    elif basis == "Y":
        for wire in wires:
            qml.adjoint(qml.S(wires=wire))
            qml.Hadamard(wires=wire)

    return qml.sample(wires=wires)

submissions = {}
rng = np.random.default_rng(1234)
for temperature in TEMPERATURES:
    params = optimized_params[temperature]
    latent_probabilities = np.asarray(latent_probability_distribution(params[:NUM_QUBITS]))
    indices = rng.choice(2 ** NUM_QUBITS, size=N_SHOTS, replace=True, p=latent_probabilities)

    submission = {"temperature": temperature}
    for basis in ("X", "Y", "Z"):
        submission[basis] = [
            np.asarray(sample_state(params[NUM_QUBITS:], index_to_bitstring(index), basis))
            .astype(int).tolist()
            for index in indices
        ]

    submissions[temperature] = submission
    path = Path(f"submission_{temperature:.4g}.json")
    path.write_text(json.dumps(submission, indent=2) + "\n")
    print(f"Wrote {path} ({N_SHOTS} shots per basis)")

## 6. Inspect the sampled states

The following plots are diagnostics rather than part of optimization. Connected correlations show how much two sites correlate beyond the product of their individual means. Magnetization histograms provide an intuitive view of the sampled states in each measurement basis.

In [ ]:
def correlations_vs_distance(shots, lattice_side=4):
    """Group connected spin correlations by periodic distance."""
    shots = np.asarray(shots, dtype=int)
    if shots.ndim != 2 or shots.shape[1] != lattice_side ** 2:
        raise ValueError(f"expected (shots, {lattice_side ** 2}) bits, got {shots.shape}")

    spins = 1.0 - 2.0 * shots
    means = spins.mean(axis=0)
    grouped = {}
    for first in range(lattice_side ** 2):
        row1, column1 = divmod(first, lattice_side)
        for second in range(first + 1, lattice_side ** 2):
            row2, column2 = divmod(second, lattice_side)
            dx = min(abs(column1 - column2), lattice_side - abs(column1 - column2))
            dy = min(abs(row1 - row2), lattice_side - abs(row1 - row2))
            distance = float(np.sqrt(dx * dx + dy * dy))
            value = np.mean(spins[:, first] * spins[:, second]) - means[first] * means[second]
            grouped.setdefault(distance, []).append(value)

    distances = np.array(sorted(grouped))
    values = np.array([np.mean(grouped[d]) for d in distances])
    errors = np.array([np.std(grouped[d], ddof=1) / np.sqrt(len(grouped[d]))
                       if len(grouped[d]) > 1 else 0.0 for d in distances])
    return distances, values, errors


fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, basis in zip(axes, ("X", "Y", "Z")):
    distance, correlation, error = correlations_vs_distance(submissions[TEMPERATURES[0]][basis])
    axis.errorbar(distance, correlation, yerr=error, fmt="o-", capsize=3)
    axis.axhline(0.0, color="gray", linestyle=":")
    axis.set_title(f"{basis} connected correlations")
    axis.set_xlabel("Periodic distance")
    axis.set_ylabel("C(r)")
fig.tight_layout()
plt.show()

## 7. Score a generated submission

Run this final cell after generating the JSON files. It scores the selected temperature file with the repository's challenge scorer and saves the detailed result.

In [ ]:
import sys

# Make the repository scorer available even when the notebook runs from CIRAQLE/.
project_dir = Path.cwd().parent if Path.cwd().name == "CIRAQLE" else Path.cwd()
sys.path.insert(0, str(project_dir))
from score import score_submission

submission_path = Path(f"submission_{TEMPERATURES[-1]:.4g}.json")
submission = json.loads(submission_path.read_text())
results = score_submission(submission)
output_path = Path("submission_results.json")
output_path.write_text(json.dumps(results, indent=2) + "\n")
print(f"Results saved to: {output_path.resolve()}")
print(json.dumps(results, indent=2))